# Off-market · standalone XGBoost × CatBoost × LightGBM blends

**New hypothesis:** mixing CatBoost and LightGBM with **one strong standalone XGBoost** may help even if adding them to our already-diversified frozen XGBoost ensemble does not.

This experiment trains standalone tuned XGBoost, CatBoost and LightGBM **on the same earlier completed sales** with the same 25,000-row/350-tree pilot budget, using the same spline/geographic off-market-safe feature family. All models predict exactly the historical properties in the frozen ensemble's saved audit file.

We report **two baselines**: (1) standalone XGBoost at identical training budget and (2) the unchanged, full-budget frozen ensemble. The first isolates the benefit of combining model families; the second tells us if the result is useful for the competition. Neither the frozen ensemble nor its saved audit file is modified.

This notebook can **reuse your completed CatBoost and LightGBM pilot prediction files**, reducing the new run to one standalone XGBoost fit per month (plus instantaneous geometric blends).

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / "offmarket_crossfamily_blend.py").is_file():
    raise FileNotFoundError("Open this root notebook from the repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from offmarket_crossfamily_blend import REFERENCE, BLENDS, run_offmarket_blend
print("Repository:",ROOT)
print("Frozen reference:",REFERENCE)
if not REFERENCE.is_file():
    print("REQUIRED: copy the historical saved shadow_audit_predictions.csv or set REFERENCE_FILE below.")
display(pd.DataFrame([
    {"blend":name, "weights":", ".join(f"{model} {w:.0%}" for model,w in weights.items())}
    for name,weights in BLENDS.items()
]))

Repository: C:\Users\deant\Desktop\Coding\Data Dash Case Comp
Frozen reference: outputs\ensemble_release_candidate_full\shadow_audit_predictions.csv


,blend,weights
0,frozen_plus_cat_80_20,"frozen_xgb_ensemble 80%, catboost_offmarket 20%"
1,frozen_plus_lgb_80_20,"frozen_xgb_ensemble 80%, lightgbm_offmarket 20%"
2,frozen_cat_lgb_70_20_10,"frozen_xgb_ensemble 70%, catboost_offmarket 20..."
3,frozen_cat_lgb_60_25_15,"frozen_xgb_ensemble 60%, catboost_offmarket 25..."


## 1 · Quick, matched-budget mixed-family screen

Train tuned standalone XGBoost using the previous pilot's **25,000 earlier sales, 350 trees, 350 homes per month**. CatBoost and LightGBM use the same budget and feature stack. If the original pilot results live under `outputs/offmarket_crossfamily_pilot/`, the new script verifies their source manifest, dates, budgets, identifiers, and observed prices before reusing them.

The frozen ensemble remains a stronger **full-budget historical reference**, not an equal-budget individual model.

In [ ]:
REFERENCE_FILE = Path(REFERENCE)
RUN_PILOT = True
PILOT_OUTPUT = Path("outputs/offmarket_base_xgb_pilot")  # New folder: preserves old scores/caches.

# Reuse historical CatBoost & LightGBM predictions ONLY if a verified matching
# source manifest and their saved per-month CSVs are available.
OLD_PILOT_OUTPUT = Path("outputs/offmarket_crossfamily_pilot")
REUSE_FAMILY_CACHES = OLD_PILOT_OUTPUT if (OLD_PILOT_OUTPUT / "manifest.json").is_file() else None

if RUN_PILOT:
    if not REFERENCE_FILE.is_file():
        raise FileNotFoundError(f"Missing frozen historical ensemble audit: {REFERENCE_FILE}")
    pooled, monthly, predictions = run_offmarket_blend(
        full=False, reference_file=REFERENCE_FILE,
        output_dir=PILOT_OUTPUT, threads=4, bootstrap_repeats=300,
        reuse_family_caches=REUSE_FAMILY_CACHES,
    )
else:
    pooled = pd.read_csv(PILOT_OUTPUT / "pooled_scores.csv")
    monthly = pd.read_csv(PILOT_OUTPUT / "monthly_scores.csv")
    predictions = pd.read_csv(PILOT_OUTPUT / "predictions.csv", dtype={"ml_num": str})

display(pooled[[
    "arm", "variant", "homes", "mdape", "mape", "p90_ape",
    "delta_base_xgb_mdape_pp", "delta_base_xgb_mape_pp",
    "delta_mdape_pp", "delta_mape_pp",
    "base_xgb_delta_mdape_ci_low", "base_xgb_delta_mdape_ci_high",
]].round(4))
print("Negative 'delta_base_xgb' = improvement vs equally-budgeted XGBoost.")
print("Negative unprefixed delta = improvement vs frozen full-budget ensemble.")

In [ ]:
import matplotlib.pyplot as plt
print("Monthly MdAPE difference versus SAME-BUDGET standalone XGBoost:")
display(monthly.pivot_table(
    index="variant", columns="month", values="delta_base_xgb_mdape_pp"
).round(3))
print("Monthly MdAPE difference versus FROZEN XGBoost ensemble:")
display(monthly.pivot_table(
    index="variant", columns="month", values="delta_mdape_pp"
).round(3))
plot = pooled.sort_values("delta_base_xgb_mdape_pp")
fig, ax = plt.subplots(figsize=(10, max(4, 0.40 * len(plot))))
ax.barh(plot.variant, plot.delta_base_xgb_mdape_pp)
ax.axvline(0, color="gray", linewidth=1)
ax.invert_yaxis()
ax.set_xlabel("Change in MdAPE vs standalone tuned XGBoost (pp; negative = better)")
ax.set_title("Same-budget cross-family blend benefit")
plt.tight_layout()
plt.show()

## 2 · Optional full-strength evaluation

If cross-family blends beat the **standalone XGBoost pilot**, `RUN_FULL=True` retrains each individual model on all earlier eligible sales with tuned boosting rounds (XGBoost 700, CatBoost 1,400, LightGBM 1,800). It compares their blends against the frozen ensemble on all available 2025 audit homes.

The old frozen ensemble uses a different architecture and feature-bagging scheme. Even full runs are exploratory because these months have already been examined.

In [ ]:
RUN_FULL = False
FULL_OUTPUT = Path("outputs/offmarket_base_xgb_full")
if RUN_FULL:
    full_pooled, full_monthly, _ = run_offmarket_blend(
        full=True, reference_file=REFERENCE_FILE,
        output_dir=FULL_OUTPUT, threads=4, bootstrap_repeats=500
    )
    display(full_pooled[[
        "variant", "mdape", "mape", "p90_ape",
        "delta_base_xgb_mdape_pp", "delta_mdape_pp",
    ]].round(4))
else:
    print("Full-strength training disabled; set RUN_FULL=True only if the pilot warrants it.")

## Decision

There are **two different questions**.

1. **Does model-family diversity help an individual XGBoost?** Compare `base_xgb_plus_cat_80_20`, `base_xgb_plus_lgb_80_20` and `base_xgb_cat_lgb_70_20_10` against `xgboost_offmarket` using the **delta_base_xgb** columns.
2. **Does any blend beat our current frozen ensemble?** Compare the same models against `frozen_xgb_ensemble` using the unprefixed deltas.

A blend can answer *yes* to the first question and *no* to the second. If that happens, it demonstrates useful diversity, but **does not justify replacing the frozen ensemble**.

These historical months are development evidence, not untouched confirmation. All weights were predeclared; no validation-driven weight optimization is performed.